# Process PRISM secondary screen

In [1]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

CWD set to: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project


imports + paths + file check

In [2]:
from pathlib import Path
import pandas as pd

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

prism_path = RAW / "secondary-screen-dose-response-curve-parameters.csv"
prism_path.exists(), prism_path

(True,
 WindowsPath('data/raw/secondary-screen-dose-response-curve-parameters.csv'))

read + inspect columns

In [3]:
pr = pd.read_csv(prism_path)
pr.shape, pr.columns[:25]

C:\Users\josha\AppData\Local\Temp\ipykernel_39752\899399137.py:1: DtypeWarning: Columns (0: disease.area, 1: indication) have mixed types. Specify dtype option on import or set low_memory=False.
  pr = pd.read_csv(prism_path)


((701004, 20),
 Index(['broad_id', 'depmap_id', 'ccle_name', 'screen_id', 'upper_limit',
        'lower_limit', 'slope', 'r2', 'auc', 'ec50', 'ic50', 'name', 'moa',
        'target', 'disease.area', 'indication', 'smiles', 'phase',
        'passed_str_profiling', 'row_name'],
       dtype='str'))

normalize + prefer PR500 if present

In [4]:
if "name" in pr.columns and "compound_name" not in pr.columns:
    pr = pr.rename(columns={"name": "compound_name"})

if "row_name" in pr.columns:
    has_pr500 = pr["row_name"].astype(str).str.contains("PR500", na=False)
    if has_pr500.any():
        pr = pr.loc[has_pr500].copy()

keep = ["depmap_id", "broad_id", "compound_name", "auc", "ic50", "r2", "ec50", "slope"]
keep = [c for c in keep if c in pr.columns]
pr = pr[keep]

pr.head(), pr.shape

(    depmap_id                broad_id compound_name       auc  ic50        r2  \
 0  ACH-000879  BRD-K71847383-001-12-5    cytarabine  1.677789   NaN -0.026964   
 1  ACH-000320  BRD-K71847383-001-12-5    cytarabine  1.240300   NaN -0.147274   
 2  ACH-001145  BRD-K71847383-001-12-5    cytarabine  1.472333   NaN  0.193893   
 3  ACH-000873  BRD-K71847383-001-12-5    cytarabine  1.207160   NaN -0.005460   
 4  ACH-000855  BRD-K71847383-001-12-5    cytarabine  1.229332   NaN  0.132818   
 
            ec50     slope  
 0  8.415093e+06 -0.022826  
 1  9.643742e+00 -0.237504  
 2  2.776687e-02 -0.302937  
 3  2.654701e+00 -0.209393  
 4  5.889041e-01 -0.277530  ,
 (701004, 8))

collapse duplicates (compound × cell line)

In [5]:
num_cols = [c for c in pr.columns if c not in ["depmap_id", "broad_id", "compound_name"]]
agg = {c: "median" for c in num_cols}
agg["compound_name"] = "first"

pr2 = pr.groupby(["depmap_id", "broad_id"], as_index=False).agg(agg)

outpath = OUT / "prism_secondary_collapsed.parquet"
pr2.to_parquet(outpath, index=False)

print("Wrote:", outpath)
print("Rows:", f"{len(pr2):,}")
pr2.head()

Wrote: data\processed\prism_secondary_collapsed.parquet
Rows: 644,798


,depmap_id,broad_id,auc,ic50,r2,ec50,slope,compound_name
0,ACH-000007,BRD-A00758722-001-04-9,0.903644,3.594623,0.272739,3.059164,3.457503,noretynodrel
1,ACH-000007,BRD-A02180903-001-04-5,0.938537,NaN,0.021503,0.125394,1.863805,betamethasone
2,ACH-000007,BRD-A03216249-003-24-3,1.457191,NaN,0.050635,0.000446,-0.226460,mepivacaine
3,ACH-000007,BRD-A03506276-001-01-5,0.512050,0.081393,0.977636,0.080971,6.584783,XL888
4,ACH-000007,BRD-A03623303-045-09-5,1.541743,NaN,-0.012929,0.000176,0.018727,metoprolol
